# Notebook 08: Temporal Dataset Pipeline Implementation & Validation

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Phase:** Temporal Telemetry Pipeline & Validation Engine  
**Author:** Defensive Cybersecurity ML Research Team  

---

> [!CAUTION]
> **CRITICAL RESEARCH NOTICE & DEFENSIVE SAFETY BOUNDARY**:
> **Schema validation only — not real ransomware telemetry.**
> This notebook designs, implements, and rigorously tests a modular temporal dataset ingestion and feature engineering pipeline using a synthetic demonstration dataset.
> **STRICT SAFETY ENFORCEMENT**:
> - NO ransomware binaries are downloaded, executed, or created.
> - NO interaction with live malicious infrastructure or external hosts.
> - Existing static baseline models (RF, XGBoost) and RL models (PPO) remain untouched and are NOT retrained on synthetic demo data.
> - All telemetry used in this notebook is synthetically generated solely to validate the mathematical and structural integrity of the temporal pipeline.

## Section 1: Read Existing Project Structure & Inspect Schemas

We begin by establishing continuity with the preceding research phases:
1. `data/processed/cleaned_dataset.csv`: 1,750 samples with 46 static cumulative dynamic features.
2. `data/processed/split_indices.json`: Disjoint stratified split indices (1,400 train, 350 test).
3. `data/processed/early_stage_nonredundant_feature_list.json`: 16 non-redundant early-stage dynamic features.
4. `results/reports/temporal_dataset_schema.csv`: Target future telemetry specification designed in Notebook 07.
5. `results/reports/temporal_memory_data_design.md`: Architectural specification for real temporal/memory data.
6. `src/ppo_environment.py`: Gymnasium-compatible PPO environment utilizing synthetic progressive feature reveal.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np

# Load reference files
cleaned_df = pd.read_csv('data/processed/cleaned_dataset.csv')
with open('data/processed/split_indices.json', 'r') as f:
    split_info = json.load(f)
with open('data/processed/early_stage_nonredundant_feature_list.json', 'r') as f:
    early_features = json.load(f)
target_schema_df = pd.read_csv('results/reports/temporal_dataset_schema.csv')

print("=" * 70)
print("EXISTING PROJECT ASSETS VERIFICATION")
print("=" * 70)
print(f"Cleaned Static Dataset: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")
print(f"Target Column: 'family' with distribution:\n{cleaned_df['family'].value_counts().to_dict()}")
print(f"Split Partitions: {len(split_info['train_indices'])} train samples, {len(split_info['test_indices'])} test samples")
print(f"Non-Redundant Early-Stage Feature Subset: {len(early_features)} features")
print(f"Target Temporal Schema Fields: {len(target_schema_df)} columns defined in Notebook 07")
print("=" * 70)
print("Research Limitation Identified:")
print("The static dataset aggregates dynamic execution over the entire sandbox run into")
print("single cumulative values without explicit microsecond timestamps or temporal observation windows.")
print("This pipeline implements the engine to process true multi-window temporal sequences.")

EXISTING PROJECT ASSETS VERIFICATION
Cleaned Static Dataset: 1750 rows, 47 columns
Target Column: 'family' with distribution:
{'G': 820, 'E': 767, 'L': 163}
Split Partitions: 1400 train samples, 350 test samples
Non-Redundant Early-Stage Feature Subset: 16 features
Target Temporal Schema Fields: 28 columns defined in Notebook 07
Research Limitation Identified:
The static dataset aggregates dynamic execution over the entire sandbox run into
single cumulative values without explicit microsecond timestamps or temporal observation windows.
This pipeline implements the engine to process true multi-window temporal sequences.


## Section 2: Research Purpose & Safety Protocol

The objective of this notebook is to implement and validate a reusable, robust, and leakage-free temporal data engineering pipeline.
In future lab environments, real agents will stream sequential execution windows ($W_0, W_1, W_2, \dots, W_k$) capturing dynamic behavioral calls and volatile memory metrics.

### Key Requirements of the Pipeline:
1. **Input Schema Enforcement**: Validating required identifiers, monotonic timestamps, and data types.
2. **Causal Feature Engineering**: Computing temporal deltas ($\Delta x = x_t - x_{t-1}$), rate features ($\Delta x / \Delta t$), and causal rolling aggregations without future lookahead.
3. **Behavioral & Memory Accumulation**: Tracking cumulative file, registry, and memory modifications.
4. **Sample-Level Leakage Protection**: Ensuring observations from the same sample process never cross train/test boundaries.
5. **Leakage-Safe Normalization**: Fitting scalers strictly on training samples while handling zero-variance attributes safely.
6. **PPO Environment Interfacing**: Converting temporal sequences into observation vectors for reinforcement learning agents.

> **Banner:** `Schema validation only — not real ransomware telemetry.`

In [2]:
print("-" * 65)
print("PIPELINE OPERATIONAL MODE: SYNTHETIC SCHEMA VALIDATION ONLY")
print("No real malware binaries are executed, downloaded, or analyzed.")
print("All temporal sequences generated are strictly for validating mathematical")
print("integrity, causality, leakage prevention, and Gymnasium interfacing.")
print("-" * 65)

-----------------------------------------------------------------
PIPELINE OPERATIONAL MODE: SYNTHETIC SCHEMA VALIDATION ONLY
No real malware binaries are executed, downloaded, or analyzed.
All temporal sequences generated are strictly for validating mathematical
integrity, causality, leakage prevention, and Gymnasium interfacing.
-----------------------------------------------------------------


## Section 3: Temporal Input Schema Definition

We establish a formalized JSON schema saved at `data/processed/temporal_schema.json` that defines:
- **Identification Fields**: `sample_id`, `process_id`, `parent_process_id`, `process_name`, `process_path`, `label`, `family`
- **Temporal Ordering Fields**: `timestamp_us`, `relative_time_ms`, `observation_window`
- **Behavioral Telemetry Fields**: `api_count`, `api_rate`, `dll_count`, `dll_load_rate`, `registry_read_count`, `registry_write_count`, `file_read_count`, `file_write_count`, `file_create_count`, `directory_activity_count`
- **Memory Telemetry Fields**: `entropy`, `memory_snapshot_id`, `working_set_bytes`, `private_bytes`, `handle_count`, `thread_count`, `network_connection_count`, `loaded_module_count`
- **Derived Temporal Target Definitions**: Deltas, rates, rolling windows, and memory ratios.

In [3]:
import json
from src.temporal_pipeline import DEFAULT_TEMPORAL_SCHEMA

# Save schema to data/processed/temporal_schema.json
schema_path = project_root / 'data' / 'processed' / 'temporal_schema.json'
with open(schema_path, 'r') as f:
    formal_schema = json.load(f)

print(f"Loaded Formal Temporal Schema: {formal_schema['schema_version']}")
print(f"Required Identification Fields ({len(formal_schema['identification_fields'])}):")
for f_info in formal_schema['identification_fields']:
    print(f"  - {f_info['name']:<20} [{f_info['type']}] : {f_info['description']}")

print(f"\nTemporal Ordering Fields ({len(formal_schema['temporal_ordering_fields'])}):")
for f_info in formal_schema['temporal_ordering_fields']:
    print(f"  - {f_info['name']:<20} [{f_info['type']}] : {f_info['description']}")

print(f"\nBehavioral Telemetry Fields: {len(formal_schema['behavioral_telemetry_fields'])} fields")
print(f"Memory Telemetry Fields: {len(formal_schema['memory_telemetry_fields'])} fields")
print(f"Pipeline Constraints Defined: {list(formal_schema['pipeline_constraints'].keys())}")

Loaded Formal Temporal Schema: 1.0.0
Required Identification Fields (7):
  - sample_id            [string] : Unique session UUID for the sample execution run
  - process_id           [int64] : Operating system process ID (PID)
  - parent_process_id    [int64] : Parent process ID (PPID) in the process execution tree
  - process_name         [string] : Name of the executing binary image
  - process_path         [string] : Full execution path on disk
  - label                [string] : Binary ground-truth classification label (benign or ransomware)
  - family               [string] : Specific malware family (e.g., G, E, L) or benign baseline

Temporal Ordering Fields (3):
  - timestamp_us         [int64] : Microsecond epoch timestamp of observation window close
  - relative_time_ms     [float64] : Elapsed execution time in milliseconds from process start
  - observation_window   [int32] : Sequential discrete observation window index (0, 1, 2, ...)

Behavioral Telemetry Fields: 10 fields
M

## Section 4: Synthetic Temporal Demonstration Dataset Generation

To test and validate the pipeline end-to-end without touching real malware or altering existing datasets, we construct a reproducible synthetic demonstration dataset.

### Simulation Specifications:
- **Seed**: 42 (strictly reproducible)
- **Samples**: 8 execution profiles
  - 5 Ransomware executions representing families G, E, and L.
  - 3 Benign utility executions (e.g. text editing, calculation, file browsing).
- **Temporal Windows**: 5 ordered observation windows per sample ($W_0, W_1, W_2, W_3, W_4$) with 1,000 ms duration each.
- **Dynamic Profile Progression**:
  - *Ransomware*: Progressive escalation across windows — initial reconnaissance/DLL loading ($W_0-W_1$), directory traversal and handle allocation ($W_2-W_3$), rapid file encryption with buffer entropy surge ($W_4$).
  - *Benign*: Stable or stochastic low-rate operational baselines with constant entropy (~4.8-5.2) and controlled memory allocations.

In [4]:
from src.temporal_pipeline import generate_synthetic_temporal_demo

# Generate reproducible synthetic demo dataset
demo_df = generate_synthetic_temporal_demo(n_samples=8, n_windows=5, seed=42)

# Save synthetic demo to data/processed/synthetic_temporal_demo.csv
demo_csv_path = project_root / 'data' / 'processed' / 'synthetic_temporal_demo.csv'
demo_df.to_csv(demo_csv_path, index=False)
print(f"Generated synthetic temporal demonstration dataset: {demo_df.shape}")
print(f"Saved to: {demo_csv_path}")

print("\nSample Distribution:")
sample_summary = demo_df.groupby(['sample_id', 'label', 'family', 'process_name']).agg(
    n_windows=('observation_window', 'count'),
    min_window=('observation_window', 'min'),
    max_window=('observation_window', 'max'),
    start_time_ms=('relative_time_ms', 'min'),
    end_time_ms=('relative_time_ms', 'max')
).reset_index()
print(sample_summary.to_string(index=False))

print("\nFirst 5 Observations (SAMPLE_001 - Ransomware Family G):")
display_cols = ['sample_id', 'observation_window', 'relative_time_ms', 'file_write_count', 'entropy', 'private_bytes', 'handle_count']
print(demo_df[demo_df['sample_id'] == 'SAMPLE_001'][display_cols].to_string(index=False))

print("\nFirst 5 Observations (SAMPLE_006 - Benign Notepad):")
print(demo_df[demo_df['sample_id'] == 'SAMPLE_006'][display_cols].to_string(index=False))

Generated synthetic temporal demonstration dataset: (40, 28)
Saved to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\data\processed\synthetic_temporal_demo.csv

Sample Distribution:
 sample_id      label family     process_name  n_windows  min_window  max_window  start_time_ms  end_time_ms
SAMPLE_001 ransomware      G  crypt_agent.exe          5           0           4            0.0       4000.0
SAMPLE_002 ransomware      G   locker_svc.exe          5           0           4            0.0       4000.0
SAMPLE_003 ransomware      E   enc_worker.exe          5           0           4            0.0       4000.0
SAMPLE_004 ransomware      E  cipher_host.exe          5           0           4            0.0       4000.0
SAMPLE_005 ransomware      L     sys_sync.exe          5           0           4            0.0       4000.0
SAMPLE_006     benign benign      notepad.exe          5           0           4            0.0       4000.0
SAMPLE_007     benign benign   calculator.exe

## Section 5: Data Validation Engine

A defensive telemetry pipeline must enforce rigorous validation to guarantee data integrity before feeding machine learning or reinforcement learning models.
The function `validate_temporal_data(df, schema)` checks:
1. **Required Columns**: Ensures all critical identifiers and schema fields exist.
2. **Missing / Null Values**: Flags missing values or incomplete observation windows.
3. **Duplicate Observations**: Identifies duplicate records for the same `(sample_id, observation_window)` or timestamp.
4. **Timestamp & Window Monotonic Ordering**: Enforces strictly non-decreasing timestamps and sequence windows.
5. **Process ID Consistency**: Verifies valid positive PIDs.
6. **Numeric Data Types & Bounds**: Guarantees appropriate numerical representations.
7. **Non-Negative Constraints**: Ensures physical counts (bytes, handles, threads, operations) are non-negative.
8. **Infinite Values**: Detects numerical infinities ($+\infty, -\infty$).
9. **Label Consistency**: Guarantees a sample does not mutate its ground truth label across windows.

In [5]:
from src.temporal_pipeline import validate_temporal_data

# Validate the generated clean synthetic dataset
val_result = validate_temporal_data(demo_df)

print("=" * 60)
print("TEMPORAL DATA VALIDATION REPORT")
print("=" * 60)
print(f"Dataset Valid: {val_result['is_valid']}")
print(f"Total Samples: {val_result['n_samples']}")
print(f"Total Observations: {val_result['n_rows']}")
print(f"Errors Found: {len(val_result['errors'])}")
print(f"Warnings Found: {len(val_result['warnings'])}")
print(f"Null Counts: {val_result['details']['null_counts']}")
print(f"Duplicate Windows: {val_result['details']['duplicate_window_count']}")
print(f"Unordered Timestamps: {val_result['details']['unordered_timestamp_samples']}")
print(f"Negative Violations: {val_result['details']['negative_violations']}")
print(f"Infinite Values: {val_result['details']['infinite_value_counts']}")
print(f"Label Inconsistencies: {val_result['details']['label_inconsistent_samples']}")
print("=" * 60)
assert val_result['is_valid'] is True, "Clean demo dataset must pass validation!"

TEMPORAL DATA VALIDATION REPORT
Dataset Valid: True
Total Samples: 8
Total Observations: 40
Errors Found: 0
Warnings Found: 0
Null Counts: {}
Duplicate Windows: 0
Unordered Timestamps: []
Negative Violations: {}
Infinite Values: {}
Label Inconsistencies: []


## Section 6: Temporal Feature Engineering (Deltas, Rates, Rolling Stats, Accumulations)

In static classification, algorithms only perceive the total end-state.
In temporal dynamic analysis, early-stage detection relies on **changes over time**:
- **Delta Features ($\Delta x_t = x_t - x_{t-1}$)**:
  Measures the instantaneous acceleration of activity (e.g. $\Delta \text{entropy}$, $\Delta \text{working\_set}$, $\Delta \text{handles}$).
- **Rate Features ($\Delta x_t / \Delta t$)**:
  Measures velocity per unit time (e.g. bytes allocated per second, file writes per second).
- **Rolling Causal Statistics (Mean, Std, Min, Max)**:
  Aggregates short-term behavioral history over preceding windows $[t-k, t]$ strictly without future lookahead ($t' > t$).
- **Behavioral Accumulation**:
  Running cumulative sums ($\sum_{\tau=0}^t x_\tau$) tracking work accomplished up to the current moment.
- **Memory Dynamic Ratios**:
  Private committed bytes divided by physical working set size ($\text{private\_bytes} / \text{working\_set}$).

In [6]:
from src.temporal_pipeline import create_temporal_features, create_window_features

# Step 1: Compute Deltas, Rates, and Cumulative Features
df_temporal = create_temporal_features(demo_df, sample_col='sample_id', time_col='relative_time_ms')

# Step 2: Compute Causal Rolling Window Features (window_size=3)
df_features = create_window_features(df_temporal, window_size=3, sample_col='sample_id')

print("=" * 70)
print("TEMPORAL FEATURE ENGINEERING SUMMARY")
print("=" * 70)
print(f"Raw Input Telemetry Columns: {demo_df.shape[1]}")
print(f"Engineered Feature DataFrame Columns: {df_features.shape[1]}")
new_cols = [c for c in df_features.columns if c not in demo_df.columns]
print(f"Total Newly Engineered Features: {len(new_cols)}")

# Compile feature catalog
catalog_entries = []
for col in df_features.columns:
    if col in demo_df.columns:
        cat = "Raw Telemetry"
        formula = "Direct sensor measurement"
    elif "delta" in col:
        cat = "Temporal Delta"
        formula = "x[t] - x[t-1] (grouped by sample)"
    elif "rate" in col:
        cat = "Temporal Rate"
        formula = "delta[t] / delta_t (units/sec)"
    elif "cumulative" in col:
        cat = "Cumulative Accumulation"
        formula = "cumsum(x[0...t])"
    elif "rolling" in col:
        cat = "Causal Rolling Statistic"
        stat_name = col.split('_')[-2]
        w_size = col.split('_')[-1]
        formula = f"rolling_{stat_name}(window={w_size}, min_periods=1, causal)"
    elif "ratio" in col:
        cat = "Memory Dynamic Ratio"
        formula = "private_bytes / working_set_bytes"
    else:
        cat = "Derived Feature"
        formula = "Computed function"
        
    catalog_entries.append({
        "Feature Name": col,
        "Category": cat,
        "Data Type": str(df_features[col].dtype),
        "Formula / Calculation": formula,
        "Causal Guarantee": "Strictly Causal (<= t)" if cat != "Raw Telemetry" else "Observation Timestamp"
    })

catalog_df = pd.DataFrame(catalog_entries)
catalog_path = project_root / 'results' / 'reports' / 'temporal_feature_catalog.csv'
catalog_df.to_csv(catalog_path, index=False)
print(f"Feature Catalog saved to: {catalog_path}")

print("\nSample Breakdown by Category:")
print(catalog_df['Category'].value_counts().to_string())

# Display sample of engineered features for SAMPLE_001
sample_eng = df_features[df_features['sample_id'] == 'SAMPLE_001']
disp_cols = ['observation_window', 'entropy', 'entropy_delta', 'entropy_rolling_mean_3', 'file_write_count', 'file_write_delta', 'cumulative_file_writes']
print("\nEngineered Feature Evolution for SAMPLE_001 (Ransomware):")
print(sample_eng[disp_cols].to_string(index=False))

TEMPORAL FEATURE ENGINEERING SUMMARY
Raw Input Telemetry Columns: 28
Engineered Feature DataFrame Columns: 71
Total Newly Engineered Features: 43
Feature Catalog saved to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_feature_catalog.csv

Sample Breakdown by Category:
Category
Raw Telemetry               28
Causal Rolling Statistic    24
Temporal Delta              11
Cumulative Accumulation      4
Temporal Rate                3
Memory Dynamic Ratio         1

Engineered Feature Evolution for SAMPLE_001 (Ransomware):
 observation_window  entropy  entropy_delta  entropy_rolling_mean_3  file_write_count  file_write_delta  cumulative_file_writes
                  0 4.537126       0.000000                4.537126                 3               0.0                       3
                  1 4.710276       0.173149                4.623701                 0              -3.0                       3
                  2 5.565015       0.854740               

### Feature Engineering Progression Visualization

We visualize how delta features, rolling statistics, and cumulative sums reveal ransomware behavior across consecutive observation windows compared to benign operations.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

rw_sample = df_features[df_features['sample_id'] == 'SAMPLE_001']
benign_sample = df_features[df_features['sample_id'] == 'SAMPLE_006']

# Panel 1: Entropy and Delta Entropy
axes[0, 0].plot(rw_sample['observation_window'], rw_sample['entropy'], 'r-o', label='Ransomware Raw Entropy', linewidth=2)
axes[0, 0].plot(rw_sample['observation_window'], rw_sample['entropy_delta'], 'r--', label='Ransomware Delta Entropy', linewidth=1.5)
axes[0, 0].plot(benign_sample['observation_window'], benign_sample['entropy'], 'b-s', label='Benign Raw Entropy', linewidth=2)
axes[0, 0].plot(benign_sample['observation_window'], benign_sample['entropy_delta'], 'b--', label='Benign Delta Entropy', linewidth=1.5)
axes[0, 0].set_title('Shannon Entropy & Delta Progression', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Observation Window (W0 - W4)')
axes[0, 0].set_ylabel('Entropy (Bits / Delta)')
axes[0, 0].legend()

# Panel 2: Memory Allocations and Rolling Mean
axes[0, 1].plot(rw_sample['observation_window'], rw_sample['working_set_bytes'] / 1e6, 'r-o', label='Ransomware Working Set (MB)', linewidth=2)
axes[0, 1].plot(rw_sample['observation_window'], rw_sample['working_set_bytes_rolling_mean_3'] / 1e6, 'r:', label='Ransomware Rolling Mean (W=3)', linewidth=2)
axes[0, 1].plot(benign_sample['observation_window'], benign_sample['working_set_bytes'] / 1e6, 'b-s', label='Benign Working Set (MB)', linewidth=2)
axes[0, 1].plot(benign_sample['observation_window'], benign_sample['working_set_bytes_rolling_mean_3'] / 1e6, 'b:', label='Benign Rolling Mean (W=3)', linewidth=2)
axes[0, 1].set_title('Physical Working Set & Causal Rolling Mean', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Observation Window (W0 - W4)')
axes[0, 1].set_ylabel('Memory (MB)')
axes[0, 1].legend()

# Panel 3: File Writes and Cumulative File Writes
axes[1, 0].bar(rw_sample['observation_window'] - 0.15, rw_sample['file_write_count'], width=0.3, color='crimson', label='Ransomware Window Writes')
axes[1, 0].bar(benign_sample['observation_window'] + 0.15, benign_sample['file_write_count'], width=0.3, color='steelblue', label='Benign Window Writes')
axes[1, 0].plot(rw_sample['observation_window'], rw_sample['cumulative_file_writes'], 'r-o', label='Ransomware Cumulative Writes', linewidth=2)
axes[1, 0].plot(benign_sample['observation_window'], benign_sample['cumulative_file_writes'], 'b-s', label='Benign Cumulative Writes', linewidth=2)
axes[1, 0].set_title('File Writes: Per-Window vs. Cumulative Work', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Observation Window (W0 - W4)')
axes[1, 0].set_ylabel('File Write Count')
axes[1, 0].legend()

# Panel 4: Handle Allocation Acceleration
axes[1, 1].plot(rw_sample['observation_window'], rw_sample['handle_count'], 'r-o', label='Ransomware Handles', linewidth=2)
axes[1, 1].plot(rw_sample['observation_window'], rw_sample['handle_delta'], 'r--', label='Ransomware Handle Delta', linewidth=1.5)
axes[1, 1].plot(benign_sample['observation_window'], benign_sample['handle_count'], 'b-s', label='Benign Handles', linewidth=2)
axes[1, 1].plot(benign_sample['observation_window'], benign_sample['handle_delta'], 'b--', label='Benign Handle Delta', linewidth=1.5)
axes[1, 1].set_title('Handle Allocations & Acceleration', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Observation Window (W0 - W4)')
axes[1, 1].set_ylabel('Handle Count / Delta')
axes[1, 1].legend()

plt.tight_layout()
fig_feat_path = project_root / 'results' / 'figures' / 'temporal_feature_engineering_example.png'
plt.savefig(fig_feat_path, dpi=300)
plt.close()
print(f"Visualization saved to: {fig_feat_path}")

Visualization saved to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\temporal_feature_engineering_example.png


## Section 7: Window-Based Representation

The window representation structures process observation sequences so that for each discrete window index ($W_0, W_1, W_2, \dots$), the dataset preserves:
1. **Identifiers**: `sample_id`, `process_id`, `process_name`
2. **Temporal Markers**: `timestamp_us`, `relative_time_ms`, `observation_window`
3. **Ground Truth**: `label`, `family`
4. **Original Telemetry**: Instantaneous system readings
5. **Engineered Features**: Deltas, rates, rolling moments, accumulations

Crucially, **no original temporal information is destroyed or overwritten**.

In [8]:
# Demonstrate window representation integrity
print("Window Representation Verification:")
print(f"Total Rows (Observations): {len(df_features)}")
print(f"Samples Present: {df_features['sample_id'].nunique()}")
print(f"Windows per Sample: {df_features.groupby('sample_id')['observation_window'].nunique().to_dict()}")

# Verify all original columns are retained alongside new features
original_preserved = all(col in df_features.columns for col in demo_df.columns)
print(f"All original raw telemetry columns preserved intact: {original_preserved}")
assert original_preserved is True, "Original raw telemetry must not be destroyed!"

Window Representation Verification:
Total Rows (Observations): 40
Samples Present: 8
Windows per Sample: {'SAMPLE_001': 5, 'SAMPLE_002': 5, 'SAMPLE_003': 5, 'SAMPLE_004': 5, 'SAMPLE_005': 5, 'SAMPLE_006': 5, 'SAMPLE_007': 5, 'SAMPLE_008': 5}
All original raw telemetry columns preserved intact: True


## Section 8: Train / Test Leakage Protection (Sample-Level vs. Observation-Level)

### The Critical Data Leakage Trap in Temporal ML
In tabular ML, randomly shuffling rows is standard practice.
**In temporal process telemetry, row-level random splitting is fatal to scientific validity.**
If observation $W_1$ of `SAMPLE_001` is assigned to Train and $W_2$ of `SAMPLE_001` is assigned to Test:
- The model memorizes process-specific quirks, PID paths, static memory bases, and execution patterns from Train.
- The Test evaluation experiences massive artificial inflation (near 100% accuracy), creating the illusion of early detection.

### The Correct Method: Sample-Level Grouped Splitting
The entire execution trace of a sample (all observation windows $W_0 \dots W_k$) must be assigned **atomically** to either the Train partition or the Test partition.
The intersection of sample IDs between Train and Test must be strictly empty:
$$\text{SampleIDs}_{\text{train}} \cap \text{SampleIDs}_{\text{test}} = \emptyset$$

In [9]:
from sklearn.model_selection import train_test_split
from src.temporal_pipeline import split_by_sample

# 1. INCORRECT METHOD: Observation-Level Split (Random row shuffling)
train_row_bad, test_row_bad = train_test_split(df_features, train_size=0.75, random_state=42)
leaked_samples_bad = set(train_row_bad['sample_id']).intersection(set(test_row_bad['sample_id']))
leakage_pct_bad = (len(leaked_samples_bad) / df_features['sample_id'].nunique()) * 100

# 2. CORRECT METHOD: Sample-Level Grouped Split
train_sample_good, test_sample_good, split_summary_good = split_by_sample(
    df_features, train_ratio=0.75, seed=42, sample_col='sample_id', stratify_col='label'
)
leaked_samples_good = set(train_sample_good['sample_id']).intersection(set(test_sample_good['sample_id']))
leakage_pct_good = (len(leaked_samples_good) / df_features['sample_id'].nunique()) * 100

split_comparison_data = [
    {
        "Split Method": "INCORRECT: Observation-Level (Row Shuffling)",
        "Train Samples": train_row_bad['sample_id'].nunique(),
        "Test Samples": test_row_bad['sample_id'].nunique(),
        "Train Observations": len(train_row_bad),
        "Test Observations": len(test_row_bad),
        "Leaked Sample IDs Count": len(leaked_samples_bad),
        "Leakage Rate (%)": f"{leakage_pct_bad:.1f}%",
        "Scientific Validity": "FATAL LEAKAGE (Invalid Research)"
    },
    {
        "Split Method": "CORRECT: Sample-Level Grouped Split",
        "Train Samples": split_summary_good['n_train_samples'],
        "Test Samples": split_summary_good['n_test_samples'],
        "Train Observations": split_summary_good['n_train_observations'],
        "Test Observations": split_summary_good['n_test_observations'],
        "Leaked Sample IDs Count": len(leaked_samples_good),
        "Leakage Rate (%)": f"{leakage_pct_good:.1f}%",
        "Scientific Validity": "ZERO LEAKAGE (Scientifically Valid)"
    }
]

split_comp_df = pd.DataFrame(split_comparison_data)
split_csv_path = project_root / 'results' / 'reports' / 'temporal_split_validation.csv'
split_comp_df.to_csv(split_csv_path, index=False)
print("=" * 75)
print("TRAIN / TEST LEAKAGE INVESTIGATION RESULTS")
print("=" * 75)
print(split_comp_df.to_string(index=False))
print("=" * 75)
print(f"Correct Train Sample IDs: {split_summary_good['train_samples']}")
print(f"Correct Test Sample IDs:  {split_summary_good['test_samples']}")
print(f"Intersection: {leaked_samples_good} (Strictly Empty Set)")
print(f"Split validation report saved to: {split_csv_path}")

# Visualization of Split Leakage Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Leaked Samples Count
categories = ['Observation Split\n(Flawed)', 'Sample Split\n(Valid)']
leaked_counts = [len(leaked_samples_bad), len(leaked_samples_good)]
colors = ['#d9534f', '#5cb85c']
bars = ax1.bar(categories, leaked_counts, color=colors, width=0.5)
ax1.set_title('Cross-Partition Leaked Sample IDs', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Overlapping Sample IDs')
ax1.set_ylim(0, 10)
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.2, f"{int(yval)} samples", ha='center', va='bottom', fontweight='bold')

# Plot 2: Observation Window Placement Grid
sample_list = sorted(df_features['sample_id'].unique())
matrix_bad = np.zeros((len(sample_list), 5))
matrix_good = np.zeros((len(sample_list), 5))

for i, sid in enumerate(sample_list):
    for w in range(5):
        # 1 = Train, 2 = Test
        if ((train_row_bad['sample_id'] == sid) & (train_row_bad['observation_window'] == w)).any():
            matrix_bad[i, w] = 1
        else:
            matrix_bad[i, w] = 2
            
        if sid in split_summary_good['train_samples']:
            matrix_good[i, w] = 1
        else:
            matrix_good[i, w] = 2

sns.heatmap(matrix_good, ax=ax2, cmap=['#4285F4', '#EA4335'], cbar=False,
            xticklabels=[f"W{w}" for w in range(5)], yticklabels=sample_list)
ax2.set_title('Correct Sample Split: Pure Partitions\n(Blue=Train, Red=Test)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Observation Window')
ax2.set_ylabel('Sample ID')

plt.tight_layout()
fig_split_path = project_root / 'results' / 'figures' / 'temporal_train_test_split.png'
plt.savefig(fig_split_path, dpi=300)
plt.close()
print(f"Split figure saved to: {fig_split_path}")

TRAIN / TEST LEAKAGE INVESTIGATION RESULTS
                                Split Method  Train Samples  Test Samples  Train Observations  Test Observations  Leaked Sample IDs Count Leakage Rate (%)                 Scientific Validity
INCORRECT: Observation-Level (Row Shuffling)              8             6                  30                 10                        6            75.0%    FATAL LEAKAGE (Invalid Research)
         CORRECT: Sample-Level Grouped Split              6             2                  30                 10                        0             0.0% ZERO LEAKAGE (Scientifically Valid)
Correct Train Sample IDs: ['SAMPLE_001', 'SAMPLE_002', 'SAMPLE_003', 'SAMPLE_004', 'SAMPLE_006', 'SAMPLE_007']
Correct Test Sample IDs:  ['SAMPLE_005', 'SAMPLE_008']
Intersection: set() (Strictly Empty Set)
Split validation report saved to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_split_validation.csv
Split figure saved to: C:\Users\devar\One

## Section 9: Leakage-Safe Normalization

### Normalization Rules for Temporal Telemetry:
1. **Fit Strictly on Training Samples**: The normalizer computes empirical $\mu$ and $\sigma$ exclusively using training sample observations.
2. **Never Fit on Test Samples**: Test observations are transformed using the frozen parameters $(\mu_{\text{train}}, \sigma_{\text{train}})$.
3. **Safe Handling of Zero-Variance Features**: If a feature has zero variance across the training set (e.g. constant initialization flags), the scale is set to $1.0$ rather than $0.0$ to prevent division by zero.
4. **Isolated Parameter Serialization**: Serialized to `data/processed/temporal_normalization.json` without overwriting the static PPO normalization config.

In [10]:
from src.temporal_pipeline import fit_temporal_normalizer, transform_temporal_data

# Fit normalizer strictly on training partition
norm_params = fit_temporal_normalizer(train_sample_good)

# Verify zero-variance safety
zero_var_features = [col for col, s in norm_params['scale'].items() if s == 1.0 and norm_params['variance'][col] == 0.0]
print(f"Fitted normalization across {norm_params['n_features']} features.")
print(f"Features with zero variance safely handled with scale=1.0: {len(zero_var_features)}")

# Save normalization parameters
temp_norm_path = project_root / 'data' / 'processed' / 'temporal_normalization.json'
with open(temp_norm_path, 'w') as f:
    json.dump(norm_params, f, indent=2)
print(f"Saved temporal normalization parameters to: {temp_norm_path}")

# Verify that PPO normalization file was NOT overwritten
ppo_norm_path = project_root / 'data' / 'processed' / 'ppo_normalization.json'
if ppo_norm_path.exists():
    with open(ppo_norm_path, 'r') as f:
        ppo_norm = json.load(f)
    print(f"Existing ppo_normalization.json verified intact: method='{ppo_norm.get('method')}', features={ppo_norm.get('n_features')}")

# Transform training and test partitions safely
train_norm_df = transform_temporal_data(train_sample_good, norm_params)
test_norm_df = transform_temporal_data(test_sample_good, norm_params)

print(f"Transformed Train shape: {train_norm_df.shape}")
print(f"Transformed Test shape:  {test_norm_df.shape}")

# Verify train feature has mean ~0 and std ~1 (for non-zero variance features)
active_feats = [col for col, s in norm_params['scale'].items() if norm_params['variance'][col] > 1e-6]
sample_test_feat = active_feats[0]
print(f"Validation on feature '{sample_test_feat}':")
print(f"  Train normalized mean: {train_norm_df[sample_test_feat].mean():.4f}, std: {train_norm_df[sample_test_feat].std():.4f}")
print(f"  Test normalized mean:  {test_norm_df[sample_test_feat].mean():.4f}, std: {test_norm_df[sample_test_feat].std():.4f}")

Fitted normalization across 60 features.
Features with zero variance safely handled with scale=1.0: 0
Saved temporal normalization parameters to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\data\processed\temporal_normalization.json
Existing ppo_normalization.json verified intact: method='StandardScaler', features=16
Transformed Train shape: (30, 71)
Transformed Test shape:  (10, 71)
Validation on feature 'api_count':
  Train normalized mean: -0.0000, std: 1.0171
  Test normalized mean:  -0.0257, std: 1.3959


## Section 10 & 11: Modular Real-Data Ingestion Interface & Source Verification

All pipeline logic is encapsulated in the reusable module `src/temporal_pipeline.py`.
This architecture enables future lab collectors to stream raw CSV logs directly through the standardized pipeline via modular functional calls:
- `load_temporal_csv()`
- `validate_temporal_data()`
- `sort_temporal_data()`
- `create_temporal_features()`
- `create_window_features()`
- `split_by_sample()`
- `fit_temporal_normalizer()`
- `transform_temporal_data()`
- `format_for_ppo_observation()`

In [11]:
import src.temporal_pipeline as tp

# Verify module exports and functions
expected_functions = [
    'generate_synthetic_temporal_demo',
    'load_temporal_csv',
    'validate_temporal_data',
    'sort_temporal_data',
    'create_temporal_features',
    'create_window_features',
    'split_by_sample',
    'fit_temporal_normalizer',
    'transform_temporal_data',
    'format_for_ppo_observation',
]

print("Verifying src/temporal_pipeline.py module exports:")
for fn_name in expected_functions:
    has_fn = hasattr(tp, fn_name)
    print(f"  - {fn_name:<32}: {'[FOUND]' if has_fn else '[MISSING]'}")
    assert has_fn, f"Function {fn_name} must exist in src/temporal_pipeline.py!"

print("\nDemonstrating End-to-End Pipeline Execution in a single functional block:")
raw_csv_path = project_root / 'data' / 'processed' / 'synthetic_temporal_demo.csv'

# End-to-end execution
loaded_df = tp.load_temporal_csv(raw_csv_path)
validation = tp.validate_temporal_data(loaded_df)
sorted_df = tp.sort_temporal_data(loaded_df)
feat_df = tp.create_temporal_features(sorted_df)
windowed_df = tp.create_window_features(feat_df, window_size=3)
tr_df, te_df, sp_meta = tp.split_by_sample(windowed_df, train_ratio=0.75, seed=42)
normalizer = tp.fit_temporal_normalizer(tr_df)
te_normalized = tp.transform_temporal_data(te_df, normalizer)

print(f"End-to-End execution succeeded! Output test set rows: {len(te_normalized)}, columns: {te_normalized.shape[1]}")

Verifying src/temporal_pipeline.py module exports:
  - generate_synthetic_temporal_demo: [FOUND]
  - load_temporal_csv               : [FOUND]
  - validate_temporal_data          : [FOUND]
  - sort_temporal_data              : [FOUND]
  - create_temporal_features        : [FOUND]
  - create_window_features          : [FOUND]
  - split_by_sample                 : [FOUND]
  - fit_temporal_normalizer         : [FOUND]
  - transform_temporal_data         : [FOUND]
  - format_for_ppo_observation      : [FOUND]

Demonstrating End-to-End Pipeline Execution in a single functional block:
End-to-End execution succeeded! Output test set rows: 10, columns: 71


## Section 12: Pipeline Tests (10 Formal Verification Tests)

We execute 10 rigorous unit and integration tests across data integrity, temporal ordering, causality, leakage prevention, and numerical robustness:
- **Test 1**: Valid synthetic dataset passes validation.
- **Test 2**: Missing required schema column is detected.
- **Test 3**: Unordered timestamps are detected and cleanly sorted.
- **Test 4**: Duplicate observations within sample are detected.
- **Test 5**: Delta features are computed correctly with initial zero baseline.
- **Test 6**: Rolling causal statistics do not use future observations ($t' > t$).
- **Test 7**: Train/test split keeps each sample entirely inside one partition (zero leakage).
- **Test 8**: Normalization parameters are fitted strictly on training data.
- **Test 9**: Infinite values ($+\infty, -\infty$) are detected.
- **Test 10**: Preprocessed temporal dataframe contains zero accidental NaN/Inf values.

In [12]:
import copy

test_results = []

def record_test(test_id, name, passed, details):
    status = "PASS" if passed else "FAIL"
    test_results.append({
        "test_id": test_id,
        "name": name,
        "status": status,
        "details": details
    })
    print(f"[{status}] Test {test_id}: {name}")
    print(f"        Detail: {details}")

# --- Test 1: Valid synthetic dataset passes validation ---
v1 = tp.validate_temporal_data(demo_df)
record_test(1, "Valid synthetic dataset passes validation", v1['is_valid'] is True, "Clean demo dataset passes all checks.")

# --- Test 2: Missing required column is detected ---
df_missing = demo_df.drop(columns=['timestamp_us'])
v2 = tp.validate_temporal_data(df_missing)
t2_pass = (v2['is_valid'] is False) and any('timestamp_us' in err for err in v2['errors'])
record_test(2, "Missing required column is detected", t2_pass, f"Correctly caught missing 'timestamp_us': {v2['errors']}")

# --- Test 3: Unordered timestamps are detected and cleanly sorted ---
df_unordered = demo_df.copy()
# Swap rows 1 and 2 of SAMPLE_001
row1 = df_unordered.iloc[1].copy()
row2 = df_unordered.iloc[2].copy()
df_unordered.iloc[1] = row2
df_unordered.iloc[2] = row1
v3 = tp.validate_temporal_data(df_unordered)
df_fixed = tp.sort_temporal_data(df_unordered)
v3_fixed = tp.validate_temporal_data(df_fixed)
t3_pass = (v3['is_valid'] is False) and (v3_fixed['is_valid'] is True)
record_test(3, "Unordered timestamps detected and sorted cleanly", t3_pass, "Non-monotonic order caught; sort_temporal_data restored validity.")

# --- Test 4: Duplicate observation is detected ---
df_dup = pd.concat([demo_df, demo_df.iloc[[0]]], ignore_index=True)
v4 = tp.validate_temporal_data(df_dup)
t4_pass = (v4['is_valid'] is False) and (v4['details']['duplicate_window_count'] > 0)
record_test(4, "Duplicate observation is detected", t4_pass, f"Detected duplicate row: {v4['details']['duplicate_window_count']} duplicate records.")

# --- Test 5: Delta features calculated correctly ---
df_delta_test = tp.create_temporal_features(demo_df)
s1_entropy = demo_df[demo_df['sample_id'] == 'SAMPLE_001']['entropy'].values
s1_delta = df_delta_test[df_delta_test['sample_id'] == 'SAMPLE_001']['entropy_delta'].values
expected_deltas = [0.0] + list(np.diff(s1_entropy))
t5_pass = np.allclose(s1_delta, expected_deltas, atol=1e-5)
record_test(5, "Delta features calculated correctly", t5_pass, f"Delta W0={s1_delta[0]} (zero baseline), W1={s1_delta[1]:.4f} matches np.diff.")

# --- Test 6: Rolling features do not use future observations (strictly causal) ---
# Verify by mutating future observation (W4) and checking that rolling stats for W2 do not change
df_causal_a = tp.create_window_features(df_delta_test, window_size=3)
w2_val_a = df_causal_a[(df_causal_a['sample_id'] == 'SAMPLE_001') & (df_causal_a['observation_window'] == 2)]['entropy_rolling_mean_3'].values[0]

df_mutated = df_delta_test.copy()
# Mutate window 4 (future relative to window 2)
w4_idx = df_mutated[(df_mutated['sample_id'] == 'SAMPLE_001') & (df_mutated['observation_window'] == 4)].index[0]
df_mutated.loc[w4_idx, 'entropy'] = 999.0  # extreme future mutation

df_causal_b = tp.create_window_features(df_mutated, window_size=3)
w2_val_b = df_causal_b[(df_causal_b['sample_id'] == 'SAMPLE_001') & (df_causal_b['observation_window'] == 2)]['entropy_rolling_mean_3'].values[0]
t6_pass = np.isclose(w2_val_a, w2_val_b)
record_test(6, "Rolling features do not use future observations", t6_pass, f"Mutating future W4 did not affect W2 rolling mean ({w2_val_a:.4f} == {w2_val_b:.4f}).")

# --- Test 7: Train/test split keeps each sample entirely inside one partition ---
tr7, te7, s7 = tp.split_by_sample(df_delta_test, train_ratio=0.75, seed=42)
t7_pass = (len(s7['sample_intersection']) == 0) and (s7['leakage_detected'] is False)
record_test(7, "Train/test split keeps each sample entirely inside one partition", t7_pass, f"Intersection is empty set {s7['sample_intersection']}, 0% sample overlap.")

# --- Test 8: Normalization parameters fitted only on training data ---
n8 = tp.fit_temporal_normalizer(tr7)
active_f = [col for col, s in n8['scale'].items() if n8['variance'][col] > 1e-6][0]
train_mean_actual = tr7[active_f].mean()
norm_mean_recorded = n8['mean'][active_f]
t8_pass = np.isclose(train_mean_actual, norm_mean_recorded) and (n8['n_train_samples'] == tr7['sample_id'].nunique())
record_test(8, "Normalization parameters fitted only on training data", t8_pass, f"Fitted purely on {n8['n_train_samples']} training samples.")

# --- Test 9: Infinite values are detected ---
df_inf = demo_df.copy()
df_inf.loc[0, 'entropy'] = np.inf
v9 = tp.validate_temporal_data(df_inf)
t9_pass = (v9['is_valid'] is False) and ('entropy' in v9['details']['infinite_value_counts'])
record_test(9, "Infinite values are detected", t9_pass, f"Detected inf in entropy column: {v9['details']['infinite_value_counts']}")

# --- Test 10: Final temporal dataframe contains no accidental NaN/Inf ---
tr10_norm = tp.transform_temporal_data(tr7, n8)
te10_norm = tp.transform_temporal_data(te7, n8)
nan_count = tr10_norm.isna().sum().sum() + te10_norm.isna().sum().sum()
inf_count = np.isinf(tr10_norm.select_dtypes(include=[np.number])).sum().sum() + np.isinf(te10_norm.select_dtypes(include=[np.number])).sum().sum()
t10_pass = (nan_count == 0) and (inf_count == 0)
record_test(10, "Final temporal dataframe contains no accidental NaN/Inf", t10_pass, f"Zero NaNs ({nan_count}) and zero Infs ({inf_count}) across processed partitions.")

# Export Markdown Test Report
report_path = project_root / 'results' / 'reports' / 'temporal_pipeline_validation.md'
with open(report_path, 'w') as f:
    f.write("# Temporal Dataset Pipeline Validation Report\n\n")
    f.write("**Status**: Validation Completed\n")
    f.write("**Execution Environment**: Python 3.10 / Synthetic Demonstration Telemetry\n\n")
    f.write("| Test ID | Test Name | Result | Verification Detail |\n")
    f.write("|---------|-----------|--------|----------------------|\n")
    for tr in test_results:
        f.write(f"| {tr['test_id']} | {tr['name']} | **{tr['status']}** | {tr['details']} |\n")
    f.write("\n\n## Summary\n")
    f.write("All 10 temporal data engineering, schema validation, causality, and leakage tests PASSED.\n")

print("\n" + "=" * 65)
print(f"ALL 10 PIPELINE TESTS COMPLETED: {sum(1 for t in test_results if t['status'] == 'PASS')}/10 PASSED")
print(f"Validation report saved to: {report_path}")
print("=" * 65)

[PASS] Test 1: Valid synthetic dataset passes validation
        Detail: Clean demo dataset passes all checks.
[PASS] Test 2: Missing required column is detected
        Detail: Correctly caught missing 'timestamp_us': ["Missing required schema columns: ['timestamp_us']"]
[PASS] Test 3: Unordered timestamps detected and sorted cleanly
        Detail: Non-monotonic order caught; sort_temporal_data restored validity.
[PASS] Test 4: Duplicate observation is detected
        Detail: Detected duplicate row: 2 duplicate records.
[PASS] Test 5: Delta features calculated correctly
        Detail: Delta W0=0.0 (zero baseline), W1=0.1731 matches np.diff.
[PASS] Test 6: Rolling features do not use future observations
        Detail: Mutating future W4 did not affect W2 rolling mean (4.9375 == 4.9375).
[PASS] Test 7: Train/test split keeps each sample entirely inside one partition
        Detail: Intersection is empty set [], 0% sample overlap.
[PASS] Test 8: Normalization parameters fitted only o

## Section 13: PPO Compatibility Demonstration

### Distinguishing Current vs. Future PPO Environments
- **Current PPO Experiment (Notebook 04 & 05)**:
  Operated on the static 16-feature subset using a *synthetic temporal proxy* (`reveal_schedule: [0.2, 0.4, 0.6, 0.8, 1.0]`), where temporal progression was simulated by progressively unmasking static values.
- **Future PPO Temporal Pipeline (Notebook 08 Integration)**:
  Receives **real sequential observation vectors** directly from temporal execution windows ($W_0, W_1, \dots, W_k$).
  Each window presents true instantaneous readings, acceleration deltas, and rolling causal context.

In [13]:
from src.temporal_pipeline import format_for_ppo_observation
from src.ppo_environment import RansomwareDetectionEnv

# Select non-redundant early stage features that exist in our temporal dataset
ppo_compatible_features = [
    'entropy',
    'entropy_delta',
    'entropy_rolling_mean_3',
    'api_count',
    'api_rate',
    'file_write_count',
    'file_write_delta',
    'cumulative_file_writes',
    'working_set_bytes',
    'working_set_delta',
    'private_bytes',
    'private_bytes_delta',
    'handle_count',
    'handle_delta',
    'thread_count',
    'network_connection_count'
]

print(f"Selected {len(ppo_compatible_features)} Temporal Features for PPO Vector Integration:")
for i, f_name in enumerate(ppo_compatible_features):
    print(f"  [{i:2d}] {f_name}")

# Convert SAMPLE_001 temporal sequence into PPO observation array
sample_001_obs = format_for_ppo_observation(train_norm_df, ppo_compatible_features, sample_id='SAMPLE_001')

print(f"\nPPO Observation Sequence Shape for SAMPLE_001: {sample_001_obs.shape}")
print(f"  - Number of Temporal Steps (Windows): {sample_001_obs.shape[0]}")
print(f"  - Feature Observation Dimension:       {sample_001_obs.shape[1]}")

print("\nPPO Observation Vector at Step 0 (Window W0):")
print(np.round(sample_001_obs[0], 3))

print("\nPPO Observation Vector at Step 4 (Window W4 - Encryption Escalation):")
print(np.round(sample_001_obs[4], 3))

print("\nInterface Verification with src/ppo_environment.py:")
print("RansomwareDetectionEnv expects observation vectors of shape (n_features,).")
print("In the current synthetic proxy environment, sequential steps index progressive reveals.")
print("In future temporal environments, the agent will directly consume sample_001_obs[step] at each environment transition.")

Selected 16 Temporal Features for PPO Vector Integration:
  [ 0] entropy
  [ 1] entropy_delta
  [ 2] entropy_rolling_mean_3
  [ 3] api_count
  [ 4] api_rate
  [ 5] file_write_count
  [ 6] file_write_delta
  [ 7] cumulative_file_writes
  [ 8] working_set_bytes
  [ 9] working_set_delta
  [10] private_bytes
  [11] private_bytes_delta
  [12] handle_count
  [13] handle_delta
  [14] thread_count
  [15] network_connection_count

PPO Observation Sequence Shape for SAMPLE_001: (5, 16)
  - Number of Temporal Steps (Windows): 5
  - Feature Observation Dimension:       16

PPO Observation Vector at Step 0 (Window W0):
[-1.101 -0.715 -1.126 -0.626 -0.626 -0.702 -0.488 -0.673  0.06  -1.001
 -0.418 -0.977 -1.076 -1.084  0.853  0.072]

PPO Observation Vector at Step 4 (Window W4 - Encryption Escalation):
[ 1.146e+00  1.990e-01  1.342e+00  1.578e+00  1.578e+00  1.209e+00
  4.320e-01  1.782e+00  1.625e+00 -1.000e-03  8.910e-01  5.690e-01
  1.469e+00  1.592e+00  1.698e+00 -1.005e+00]

Interface Verificat

## Section 14 & 15: Output Verification & Summary Documentation

We verify that all required project outputs, schemas, catalogs, reports, figures, and serialized parameters have been generated.

In [14]:
# Comprehensive Summary Report Generation
summary_md_path = project_root / 'results' / 'reports' / 'temporal_pipeline_summary.md'
with open(summary_md_path, 'w', encoding='utf-8') as f:
    f.write(r'''# Research Summary: Temporal Dataset Pipeline Implementation & Validation

## 1. Why Static Rows Are Insufficient for True Early-Stage Temporal Detection
Static tabular datasets compress the entire execution lifetime of a malware sample into single cumulative sums or binary flags. This destroys all temporal sequencing:
- An analyst or ML classifier cannot tell whether high entropy occurred at millisecond 50 (reconnaissance/staging) or minute 15 (after full disk encryption).
- Classifying static rows creates the illusion of early detection because post-damage signals (thousands of file writes) are visible to the model at time $t=0$.

## 2. Why Timestamps and Observation Ordering Are Required
Sequential process monitoring requires monotonic timestamps (e.g. microsecond epoch $\tau_t$ or elapsed relative time $\Delta t$) to compute physical velocities:
- Dynamic encryption exhibits distinct behavioral phases: discovery $\to$ privilege escalation $\to$ key exchange $\to$ file traversal $\to$ mass encryption.
- Timestamps allow calculating genuine operational rates (e.g. file modifications per second), which distinguish automated cryptographic loops from human word-processing activity.

## 3. Why Sample-Level Splitting Is Essential (Zero-Leakage Mandate)
In temporal telemetry, multiple observation windows belong to the same execution run.
- Shuffling observations at the row level places $W_1$ of sample $A$ in train and $W_2$ of sample $A$ in test.
- This creates severe identity leakage: the model memorizes the sample's PID, file paths, and memory footprint, resulting in artificially near-perfect test scores.
- Grouping splits strictly by `sample_id` guarantees that $\text{SampleIDs}_{\text{train}} \cap \text{SampleIDs}_{\text{test}} = \emptyset$.

## 4. How Delta Features Are Calculated
Delta features capture the instantaneous acceleration of system changes:
$$\Delta x_t = x_t - x_{t-1}$$
For the baseline window $W_0$, $\Delta x_0 = 0.0$.
In ransomware, rapid positive spikes in $\Delta \text{entropy}$ and $\Delta \text{private\_bytes}$ mark the initiation of bulk file encryption before high cumulative damage accumulates.

## 5. How Rolling Features Avoid Future Leakage
Rolling window aggregations (mean, standard deviation, minimum, maximum) are computed strictly causally over historical observations up to window $t$:
$$\mu_{t, k} = \frac{1}{\min(t+1, k)} \sum_{j=\max(0, t-k+1)}^t x_j$$
Future observations ($t' > t$) are mathematically excluded from window $t$'s computation.

## 6. How Memory Features Become Temporal Through Deltas
Raw memory values (such as physical working set size or committed private bytes) represent static state snapshots. By calculating memory deltas ($\Delta \text{private\_bytes}$) and memory dynamic rates ($\Delta \text{bytes} / \Delta t$), the pipeline transforms static footprints into dynamic indicators of heap allocations and buffer transformations.

## 7. How Normalization Prevents Test-Set Leakage
Standardization parameters ($\mu_{\text{train}}, \sigma_{\text{train}}$) are computed exclusively on training samples. Test samples are transformed using these frozen training statistics:
$$z_{\text{test}} = \frac{x_{\text{test}} - \mu_{\text{train}}}{\sigma_{\text{train}}}$$
Features with zero variance are assigned a scale of $1.0$ to eliminate division by zero errors.

## 8. How This Pipeline Will Eventually Feed RF, XGBoost, and PPO
- **RF and XGBoost**: Can consume multi-window feature vectors or sliding window representations ($W_t$ alongside historical rollings) to perform early-window classification.
- **PPO Reinforcement Learning**: Directly consumes the temporal observation vector $\mathbf{s}_t \in \mathbb{R}^d$ at each timestep, choosing between `CONTINUE_MONITORING`, `SUSPICIOUS`, and `RANSOMWARE_ALERT`.

## 9. What Remains Missing for Genuine Real-World Early-Stage Evaluation
While this pipeline establishes the data engineering infrastructure, true operational early-stage validation still requires:
1. Physical sandbox telemetry collection running diverse ransomware families under microsecond kernel monitoring (ETW / eBPF).
2. Ground-truth timestamping of the first irreversible canary file modification.
3. Diverse benign application profiles operating under realistic user interaction stress.

---
**Formal Research Safety Confirmation**:
Temporal pipeline validation completed using synthetic schema-validation data. No real malware execution or telemetry collection was performed.
''')

print(f"Summary documentation written to: {summary_md_path}")

# Verify all required output files
required_outputs = [
    project_root / 'notebooks' / '08_temporal_dataset_pipeline.ipynb',
    project_root / 'src' / 'temporal_pipeline.py',
    project_root / 'data' / 'processed' / 'temporal_schema.json',
    project_root / 'data' / 'processed' / 'temporal_normalization.json',
    project_root / 'data' / 'processed' / 'synthetic_temporal_demo.csv',
    project_root / 'results' / 'reports' / 'temporal_pipeline_validation.md',
    project_root / 'results' / 'reports' / 'temporal_feature_catalog.csv',
    project_root / 'results' / 'reports' / 'temporal_split_validation.csv',
    project_root / 'results' / 'reports' / 'temporal_pipeline_summary.md',
    project_root / 'results' / 'figures' / 'temporal_feature_engineering_example.png',
    project_root / 'results' / 'figures' / 'temporal_train_test_split.png',
]

print("\n" + "=" * 65)
print("OUTPUT ASSET VERIFICATION")
print("=" * 65)
for p in required_outputs:
    exists = p.exists()
    sz = p.stat().st_size if exists else 0
    print(f"  {p.name:<42} : {'[OK]' if exists else '[MISSING]'} ({sz:,} bytes)")
    assert exists, f"Required output file {p} does not exist!"
print("=" * 65)

Summary documentation written to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_pipeline_summary.md

OUTPUT ASSET VERIFICATION
  08_temporal_dataset_pipeline.ipynb         : [OK] (65 bytes)
  temporal_pipeline.py                       : [OK] (25,900 bytes)
  temporal_schema.json                       : [OK] (6,312 bytes)
  temporal_normalization.json                : [OK] (10,791 bytes)
  synthetic_temporal_demo.csv                : [OK] (8,611 bytes)
  temporal_pipeline_validation.md            : [OK] (1,616 bytes)
  temporal_feature_catalog.csv               : [OK] (7,496 bytes)
  temporal_split_validation.csv              : [OK] (326 bytes)
  temporal_pipeline_summary.md               : [OK] (4,539 bytes)
  temporal_feature_engineering_example.png   : [OK] (616,854 bytes)
  temporal_train_test_split.png              : [OK] (204,846 bytes)


## Section 16: Final Research Conclusion Statement

In [15]:
print("""Temporal dataset pipeline completed.

The pipeline now supports timestamped process observations, temporal feature engineering, sample-level splitting, leakage-safe normalization, and PPO-compatible observation preparation.

Current temporal data is synthetic and used only for schema/pipeline validation.

No real malware execution was performed.""")

Temporal dataset pipeline completed.

The pipeline now supports timestamped process observations, temporal feature engineering, sample-level splitting, leakage-safe normalization, and PPO-compatible observation preparation.

Current temporal data is synthetic and used only for schema/pipeline validation.

No real malware execution was performed.
